In [ ]:
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as f, types
import findspark
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np
import os
from pathlib import Path
import geopandas as gpd
from shapely import wkt
import json
from aux_functions_wealth import *

In [ ]:
projectRoot = Path.cwd().parents[2]

CLEAN_DATA_YELLOW = projectRoot / "data" / "clean" / "clean_tlc_trip_record" / "yellow"
CLEAN_DATA_FHVHV = projectRoot / "data" / "clean" / "clean_tlc_trip_record" / "fhvhv"
GEO_DATA = projectRoot / "data" / "NYC_Taxi_Zones.csv"
TABLES_DIR = projectRoot / "data" / "plotTables" / "economy" / "zones"
HTML_DIR = projectRoot / "data" / "html" / "economy" / "zones"
OUTPUT_Y_POIS_DEMAND = projectRoot / "data" / "clean" /"wealthAnalysis" / "yellow_zone_demand_pois.parquet"
OUTPUT_FHV_POIS_DEMAND = projectRoot / "data" / "clean" /"wealthAnalysis" / "fhv_zone_demand_pois.parquet"
OUTPUT_PATH_Y  = projectRoot / "data" / "clean" / "wealthAnalysis" / "yellow_zone_demand.parquet"
OUTPUT_PATH_FHV  = projectRoot / "data" / "clean" / "wealthAnalysis" / "fhvhv_zone_demand.parquet"
RAW_POIS_PATH = projectRoot / "data" / "raw" / "pois"
CLEAN_POIS_PATH = projectRoot / "data" / "clean" / "pois"

ROLLING_SALES_OUTPUT_Y =  projectRoot / "data" / "clean" / "wealthAnalysis" / "rollingsales_nyc_yellow.csv"
ROLLING_SALES_OUTPUT_FHV =  projectRoot / "data" / "clean" / "wealthAnalysis" / "rollingsales_nyc_fhv.csv"

os.makedirs(TABLES_DIR, exist_ok=True)
os.makedirs(HTML_DIR, exist_ok=True)

## SPARK CONFIGURATION

In [ ]:
findspark.init()

#Adjust these settings based on your system's resources and requirements


spark = (SparkSession.builder
        .master("local[6]")  
        .config("spark.driver.memory", "8g")
        .config("spark.executor.memory", "8g")
        .config("spark.sql.shuffle.partitions", "100")
        .config("spark.sql.adaptive.advisoryPartitionSizeInBytes", "64m")
        .config("spark.driver.maxResultSize", "1g")
        .config("spark.sql.session.timeZone", "America/New_York")
        .config("spark.sql.timestampType", "TIMESTAMP_NTZ")
        # Network binding fixes for macOS
        .config("spark.driver.host", "127.0.0.1")
        .config("spark.driver.bindAddress", "127.0.0.1")
        .getOrCreate())

In [ ]:
def save_table(df:pd.DataFrame, name:str):
    path = os.path.join(str(TABLES_DIR), f"{name}.csv")
    df.to_csv(path, index=False)

def load_csv(path:Path) -> pd.DataFrame:
    return pd.read_csv(path)

LAYOUT_DEFAULTS = dict(
    template="plotly_white",
    font=dict(family="Inter, sans-serif", size=12),
    hovermode="x unified",
    margin=dict(l=60, r=30, t=60, b=50),
)

DAY_ORDER = ["Monday", "Tuesday", "Wednesday", "Thursday",
             "Friday", "Saturday", "Sunday"]

In [ ]:

# ZONE
df_geodata = pd.read_csv(GEO_DATA)
df_geodata = df_geodata.rename(columns ={"the_geom" : "geometry", "shape_leng" : "length", "shape_area" : "area", "locationid" : "id"})
# Convierte WKT → geometría
df_geodata['geometry'] = df_geodata['geometry'].apply(wkt.loads) 
gdf = gpd.GeoDataFrame(df_geodata, crs='EPSG:4326')
gdf_proj = gdf.to_crs('EPSG:2263')
gdf_proj['centroid'] = gdf_proj.geometry.centroid
gdf['lat'] = gdf_proj.centroid.to_crs('EPSG:4326').y
gdf['lon'] = gdf_proj.centroid.to_crs('EPSG:4326').x

# LOAD DEMAND DATASET

In [ ]:
df_demand_yellow = spark.read.parquet(str(OUTPUT_PATH_Y))
df_demand_fhv = spark.read.parquet(str(OUTPUT_PATH_FHV))

In [ ]:
import os
import plotly.io as pio
os.environ["KALEIDO_EXECUTABLE"] = "/usr/bin/google-chrome-stable"
os.environ["KALEIDO_NO_SANDBOX"] = "1" # Vital en Linux para evitar bloqueos

In [ ]:
path_images_final = projectRoot / "data" / "images"
os.makedirs(path_images_final, exist_ok=True)


In [ ]:
import pandas as pd
import plotly.express as px
import json
from copy import copy

def generate_cloropleth_yellow_taxi_zone(df_map: pd.DataFrame, geojson: json):

    title_borough = f"Índice de poder adquisitivo por taxis en NYC"
    title_html = f"taxi_purchasing_power_patterns.html"
    fig = px.choropleth_map(
        df_map,
        geojson=geojson,
        locations=f"pickup_zone",              # column in df_map
        featureidkey="properties.zone",       # path in GeoJSON features
        color="wealth_index",
        color_continuous_scale="RdYlGn",
        map_style="carto-positron",
        zoom=10,
        center={"lat": 40.7128, "lon": -74.0060},
        opacity=0.7,
        title=title_borough,
        hover_data=["wealth_index","avg_tip_pct", "total_trips",
        "avg_daily_pickups", "median_fare"]
    )

    fig.update_layout(
        margin={"r": 0, "t": 30, "l": 0, "b": 0},
        width=1200,
        height=800,
    )

    fig.show()
    fig.write_html(HTML_DIR / title_html)
    #fig.write_image(projectRoot / "data/images/taxi_only.png", scale = 3)

def purchase_power_map_taxis(df: DataFrame, gdf: DataFrame, table_path: Path = None, option: str = 'pickup'):
    # 1. Get rid of zones with a very small amount of average daily pickups
    df_map = df_demand_yellow.filter(f.col("avg_daily_pickups") > 5.0)
    df_map = df_map.filter(~ f.col("pickup_zone").contains('Airport'))

    # 2. Dissolve zones into borough-level polygons
    gdf_borough = gdf.dissolve(by="zone").reset_index()
    geojson = json.loads(gdf_borough.to_json())

    # 3. min and max scaling; to create an index that represents wealth
    cols_select_norm = ["avg_tip_pct", "total_trips",
        "avg_daily_pickups", "median_fare"]
    for col_df in cols_select_norm:
        min_val = df_map.select(f.min(col_df)).first()[0]
        max_val = df_map.select(f.max(col_df)).first()[0]
        diff = max_val - min_val
        df_map = df_map.withColumn(f"{col_df}_norm", (f.col(col_df) - min_val) / diff)  

    df_map = df_map.withColumn("wealth_index", f.col("avg_tip_pct_norm")*0.4 +
                               f.col("avg_daily_pickups_norm") * 0.3 + f.col("median_fare_norm")* 0.3)

    # 3. Choropleth
    generate_cloropleth_yellow_taxi_zone(df_map, geojson)

purchase_power_map_taxis(df_demand_yellow, gdf)

In [ ]:
import pandas as pd
import plotly.express as px
import json
from copy import copy

def generate_cloropleth_fhv_zone(df_map: pd.DataFrame, geojson: json):

    title_borough = f"Índice de poder adquisitivo por fhv en NYC"
    title_html = f"fhv_purchasing_power_patterns.html"
    fig = px.choropleth_map(
        df_map,
        geojson=geojson,
        locations=f"pickup_zone",              # column in df_map
        featureidkey="properties.zone",       # path in GeoJSON features
        color="wealth_index",
        color_continuous_scale="RdYlGn",
        map_style="carto-positron",
        zoom=9.8,
        center={"lat": 40.7128, "lon": -74.0060},
        opacity=0.7,
        title=title_borough,
        hover_data=["wealth_index","avg_tip_pct", "total_trips",
        "avg_daily_pickups", "median_fare"]
    )

    fig.update_layout(
        margin={"r": 0, "t": 30, "l": 0, "b": 0},
        width=1200,
        height=800,
    )

    fig.show()
    fig.write_html(HTML_DIR / title_html)
    #fig.write_image(projectRoot / "data/images/uber_only.png", scale = 3)


def purchase_power_map_fhv(df: DataFrame, gdf: DataFrame, table_path: Path = None, option: str = 'pickup'):
    # 1. Get rid of zones with a very small amount of average daily pickups
    df_map = df_demand_fhv.filter(f.col("avg_daily_pickups") > 5.0)
    df_map = df_map.filter(~ f.col("pickup_zone").contains('Airport'))

    # 2. Dissolve zones into borough-level polygons
    gdf_borough = gdf.dissolve(by="zone").reset_index()
    geojson = json.loads(gdf_borough.to_json())

    # 3. min and max scaling; to create an index that represents wealth
    cols_select_norm = ["avg_tip_pct", "total_trips",
        "avg_daily_pickups", "median_fare"]
    for col_df in cols_select_norm:
        min_val = df_map.select(f.min(col_df)).first()[0]
        max_val = df_map.select(f.max(col_df)).first()[0]
        diff = max_val - min_val
        df_map = df_map.withColumn(f"{col_df}_norm", (f.col(col_df) - min_val) / diff)  

    df_map = df_map.withColumn("wealth_index", f.col("avg_tip_pct_norm")*0.2 +
                               f.col("avg_daily_pickups_norm") * 0.3 + f.col("median_fare_norm")* 0.5)

    # 3. Choropleth
    generate_cloropleth_fhv_zone(df_map, geojson)

purchase_power_map_fhv(df_demand_fhv, gdf)

## POIS

In [ ]:
df_demand_yellow_pois = spark.read.parquet(str(OUTPUT_Y_POIS_DEMAND))
df_demand_fhv_pois = spark.read.parquet(str(OUTPUT_FHV_POIS_DEMAND))

In [ ]:
df_demand_yellow_pois.show(5)

In [ ]:
import pandas as pd
import plotly.express as px
import json
from copy import copy

def generate_cloropleth_yellow_taxi_zone_pois(df_map: pd.DataFrame, geojson: json, name: str):

    title_borough = f"Índice de poder adquisitivo por {name} y puntos de interes en NYC"
    title_html = f"{name}_purchasing_power_patterns_pois.html"
    fig = px.choropleth_map(
        df_map,
        geojson=geojson,
        locations=f"pickup_zone",              # column in df_map
        featureidkey="properties.zone",       # path in GeoJSON features
        color="wealth_index",
        color_continuous_scale="RdYlGn",
        map_style="carto-positron",
        zoom=10,
        center={"lat": 40.7128, "lon": -74.0060},
        opacity=0.7,
        title=title_borough,
        hover_data=["wealth_index","avg_tip_pct", "trips_to_high_SES_pct", "low_SES_density",
        "avg_daily_pickups", "median_fare"]
    )

    fig.update_layout(
        margin={"r": 0, "t": 30, "l": 0, "b": 0},
        width=1200,
        height=800,
    )

    fig.show()
    #fig.write_html(HTML_DIR / title_html)
    #fig.write_image(projectRoot / f"data/images/{name}_pois.png", scale = 3)

def purchase_power_map_taxis_pois(df: DataFrame, gdf: DataFrame, table_path: Path = None, name: str = "taxi"):
    # 1. Get rid of zones with a very small amount of average daily pickups
    df_map = df.filter(f.col("avg_daily_pickups") > 5.0)
    df_map = df_map.filter(~ f.col("pickup_zone").contains('Airport'))
    df_map = df_map.filter(f.col("low_SES_density")>(-90))

    # 2. Dissolve zones into borough-level polygons
    gdf_borough = gdf.dissolve(by="zone").reset_index()
    geojson = json.loads(gdf_borough.to_json())

    # 3. min and max scaling; to create an index that represents wealth
    cols_select_norm = ['avg_daily_pickups', 'avg_tip_pct','median_fare', 'trips_to_high_SES_pct','low_SES_density'] 
    for col_df in cols_select_norm:
        min_val = df_map.select(f.min(col_df)).first()[0]
        max_val = df_map.select(f.max(col_df)).first()[0]
        diff = max_val - min_val
        df_map = df_map.withColumn(f"{col_df}_norm", (f.col(col_df) - min_val) / diff)  

    df_map = df_map.withColumn("wealth_index", f.col("avg_tip_pct_norm")*0.2 +
                               f.col("avg_daily_pickups_norm") * 0.2 + f.col("median_fare_norm")* 0.2 + 
                               f.col("trips_to_high_SES_pct_norm")* 0.2 + f.col("low_SES_density_norm")* 0.2)

    # 3. Choropleth
    generate_cloropleth_yellow_taxi_zone_pois(df_map, geojson, name)


In [ ]:
purchase_power_map_taxis_pois(df_demand_yellow_pois, gdf)

In [ ]:
purchase_power_map_taxis_pois(df_demand_fhv_pois, gdf, None, "fhv")

## HOUSE PRICES

In [ ]:
df_demand_yellow_house = spark.read.parquet(str(ROLLING_SALES_OUTPUT_Y))
df_demand_fhv_house = spark.read.parquet(str(ROLLING_SALES_OUTPUT_FHV))

In [ ]:
df_demand_fhv_house.show(5)

## ONLY COUNTING THE PRICE BY SQFT

In [ ]:
import pandas as pd
import plotly.express as px
import json
from copy import copy

def generate_cloropleth_yellow_taxi_zone_houses(df_map: pd.DataFrame, geojson: json, name: str):
    if name == None:
        title_borough = f"Índice de poder adquisitivo por precio por pie cuadrado en toda la ciudad"
        name = "NYC"
    else:
        title_borough = f"Índice de poder adquisitivo por precio por pie cuadrado {name}"
    title_html = f"house_purchasing_power_patterns_pois.html"
    fig = px.choropleth_map(
        df_map,
        geojson=geojson,
        locations=f"pickup_zone",              # column in df_map
        featureidkey="properties.zone",       # path in GeoJSON features
        color="median_price_per_sqft",
        color_continuous_scale="RdYlGn",
        map_style="carto-positron",
        zoom=10,
        center={"lat": 40.7128, "lon": -74.0060},
        opacity=0.7,
        title=title_borough,
        hover_data=["median_price_per_sqft", "mean_price_per_sqft","num_sells"]
    )

    fig.update_layout(
        margin={"r": 0, "t": 30, "l": 0, "b": 0},
        width=1200,
        height=800,
    )

    fig.show()
    #fig.write_html(HTML_DIR / title_html)
    fig.write_image(projectRoot / f"data/images/{name}_house.png", scale = 3)

def purchase_power_map_taxis_houses(df: DataFrame, gdf: DataFrame, table_path: Path = None, district = None):
    # 1. Get rid of zones with almost no sales
    df = df.filter(f.col("num_sells") > 5)
    df = df.filter(f.col("mean_price_per_sqft") < 10000)
    if district is not None:
        df = df.filter(f.col("pickup_borough") == district)

    # 2. Dissolve zones into borough-level polygons
    gdf_borough = gdf.dissolve(by="zone").reset_index()
    geojson = json.loads(gdf_borough.to_json())

    # 3. Choropleth
    generate_cloropleth_yellow_taxi_zone_houses(df, geojson, district)
purchase_power_map_taxis_houses(df_demand_yellow_house, gdf)    


In [ ]:
nom_b = ["Manhattan", "Queens", "Brooklyn", "Bronx", "Staten Island"]
purchase_power_map_taxis_houses(df_demand_yellow_house, gdf,None, nom_b[4])    


## TAKING EVERYTHING INTO ACCOUNT

In [ ]:
import pandas as pd
import plotly.express as px
import json
from copy import copy

def generate_cloropleth_yellow_taxi_zone_houses_pois(df_map: pd.DataFrame, geojson: json, name):

    title_borough = f"Índice de poder adquisitivo por {name} y puntos de interes en NYC"
    title_html = f"{name}_purchasing_power_patterns_pois.html"
    fig = px.choropleth_map(
        df_map,
        geojson=geojson,
        locations=f"pickup_zone",              # column in df_map
        featureidkey="properties.zone",       # path in GeoJSON features
        color="wealth_index",
        color_continuous_scale="RdYlGn",
        map_style="carto-positron",
        zoom=10,
        center={"lat": 40.7128, "lon": -74.0060},
        opacity=0.7,
        title=title_borough,
        hover_data=["wealth_index","avg_tip_pct", "trips_to_high_SES_pct", "low_SES_density",
        "avg_daily_pickups", "median_fare", "median_price_per_sqft", "num_sells"]
    )

    fig.update_layout(
        margin={"r": 0, "t": 30, "l": 0, "b": 0},
        width=1200,
        height=800,
    )

    fig.show()
    #fig.write_html(HTML_DIR / title_html)
    fig.write_image(projectRoot / f"data/images/{name}_house.png", scale = 3)

def purchase_power_map_taxis_houses_pois(df: DataFrame, gdf: DataFrame, table_path: Path = None, name = "taxi"):
    # 1. Get rid of zones with a very small amount of average daily pickups
    df_map = df.filter(f.col("avg_daily_pickups") > 5.0)
    df_map = df_map.filter(~ f.col("pickup_zone").contains('Airport'))
    df_map = df_map.filter(f.col("num_sells") > 5)

    # 2. Dissolve zones into borough-level polygons
    gdf_borough = gdf.dissolve(by="zone").reset_index()
    geojson = json.loads(gdf_borough.to_json())

    # 3. min and max scaling; to create an index that represents wealth
    cols_select_norm = ['avg_daily_pickups', 'avg_tip_pct','median_fare', 'trips_to_high_SES_pct','low_SES_density', 'median_price_per_sqft'] 
    for col_df in cols_select_norm:
        min_val = df_map.select(f.min(col_df)).first()[0]
        max_val = df_map.select(f.max(col_df)).first()[0]
        diff = max_val - min_val
        df_map = df_map.withColumn(f"{col_df}_norm", (f.col(col_df) - min_val) / diff)  

    df_map = df_map.withColumn("wealth_index", f.col("avg_tip_pct_norm")*0.1 +
                               f.col("avg_daily_pickups_norm") * 0.18 + f.col("median_fare_norm")* 0.18 + 
                               f.col("trips_to_high_SES_pct_norm")* 0.18 + f.col("low_SES_density_norm")* 0.18
                               + f.col("median_price_per_sqft")*0.18)

    # 3. Choropleth
    generate_cloropleth_yellow_taxi_zone_houses_pois(df_map, geojson, name)


In [ ]:
purchase_power_map_taxis_houses_pois(df_demand_yellow_house, gdf)

In [ ]:
purchase_power_map_taxis_houses_pois(df_demand_fhv_house, gdf, None, "fhv")